In [ ]:
# If %pip doesn't work in your Jupyter, run: pip install -q jieba sacrebleu pandas
%pip -q install jieba sacrebleu pandas

import os, re, json, math
from collections import Counter, defaultdict
from typing import List, Tuple, Dict
import pandas as pd
import jieba, sacrebleu
from pathlib import Path

CWD = Path.cwd()
print("CWD =", CWD)

# Try to find the test file anywhere under the project
test_file = None
for p in CWD.rglob("dataset_CN_EN.txt"):
    test_file = p
    break

# Try to find the SMT "best" folder (the one that has phrase_table.json)
best_dir = None
for p in CWD.rglob("phrase_table.json"):
    best_dir = p.parent      # the folder that contains phrase_table.json
    break

print("Found test_file :", test_file)
print("Found best_dir  :", best_dir)



Note: you may need to restart the kernel to use updated packages.
CWD = c:\Users\Kevin\Desktop\Artificial Intelligent\Artificial-Intelligent\model
Found test_file : None
Found best_dir  : c:\Users\Kevin\Desktop\Artificial Intelligent\Artificial-Intelligent\model\smt_runs\zh_en_pbsmt_s2t_only



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


AssertionError: Test file not found: None

In [ ]:
def tok_zh(s: str) -> List[str]:
    return [t for t in jieba.lcut(str(s)) if t.strip()]

def tok_en(s: str) -> List[str]:
    return re.findall(r"\b\w+\b", str(s).lower())

def load_test(path: str) -> Tuple[List[List[str]], List[List[str]], List[str], List[str]]:
    src_tok, ref_tok, src_raw, ref_raw = [], [], [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: 
                continue
            if "\t" in line:
                cn, en = line.split("\t", 1)
            else:
                # fallback separators
                found = False
                for sep in [" || ", " | ", "|||", " :: ", " |", "  "]:
                    if sep in line:
                        cn, en = line.split(sep, 1)
                        found = True
                        break
                if not found:
                    continue
            src_raw.append(cn.strip()); ref_raw.append(en.strip())
            src_tok.append(tok_zh(cn));  ref_tok.append(tok_en(en))
    return src_tok, ref_tok, src_raw, ref_raw


In [3]:
def load_phrase_table(best_dir: str):
    pt_path = os.path.join(best_dir, "phrase_table.json")
    with open(pt_path, "r", encoding="utf-8") as f:
        pt = json.load(f)
    phi, lex = pt.get("phi", {}), pt.get("lex", {})

    def dict_strkeys_to_tuples(d):
        out = {}
        for f_str, e_dict in d.items():
            f = tuple(f_str.split()) if f_str else tuple()
            out[f] = {}
            for e_str, v in e_dict.items():
                e = tuple(e_str.split()) if e_str else tuple()
                out[f][e] = float(v)
        return out

    return dict_strkeys_to_tuples(phi), dict_strkeys_to_tuples(lex)

def load_ibm1(best_dir: str):
    for name in ["ibm1_s2t.json", "ibm1_s2t_final.json"]:
        p = os.path.join(best_dir, name)
        if os.path.exists(p):
            data = json.load(open(p, "r", encoding="utf-8"))
            return {k: {kk: float(vv) for kk, vv in d.items()} for k, d in data.items()}
    return {}

def load_lm(best_dir: str, alpha: float):
    lm_path = None
    for name in ["lm_trigram_counts.json", "lm_counts.json"]:
        p = os.path.join(best_dir, name)
        if os.path.exists(p):
            lm_path = p; break
    if lm_path is None:
        raise FileNotFoundError("LM counts not found (lm_trigram_counts.json or lm_counts.json)")

    counts = json.load(open(lm_path, "r", encoding="utf-8"))
    unigrams = counts.get("unigrams", {})
    bigrams  = {tuple(k.split()): int(v) for k, v in counts.get("bigrams", {}).items()}
    trigrams = {tuple(k.split()): int(v) for k, v in counts.get("trigrams", {}).items()}

    BOS, EOS = "<s>", "</s>"
    total_unigrams = sum(int(c) for c in unigrams.values())
    V = max(1, len(unigrams))

    def logprob(nextw: str, w1: str, w2: str) -> float:
        tri = trigrams.get((w1, w2, nextw), 0)
        if tri > 0:
            denom = bigrams.get((w1, w2), 1)
            return math.log(tri / denom)
        bi = bigrams.get((w2, nextw), 0)
        if bi > 0:
            denom = int(unigrams.get(w2, 1))
            return math.log(alpha * bi / denom)
        uni = int(unigrams.get(nextw, 0))
        return math.log(alpha * alpha * (uni + 1) / (total_unigrams + V + 1))

    return logprob, BOS, EOS


In [4]:
from functools import lru_cache

def build_decoder(best_dir: str):
    # defaults (override with decode_meta.json if present)
    cfg = {
        "MAX_SRC_PHRASE_LEN": 8,
        "W_PHRASE": 1.0,
        "W_LEX": 1.0,
        "W_LM": 1.0,
        "W_WORD_PENALTY": -0.1,
        "MAX_JUMP": 3,
        "DIST_PENALTY": -0.2,
        "LM_ALPHA": 0.3
    }
    meta_p = os.path.join(best_dir, "decode_meta.json")
    if os.path.exists(meta_p):
        try:
            cfg.update(json.load(open(meta_p, "r", encoding="utf-8")))
        except Exception:
            pass

    phrase_table, lex_table = load_phrase_table(best_dir)
    ibm1 = load_ibm1(best_dir)
    lm_logprob, BOS, EOS = load_lm(best_dir, alpha=float(cfg["LM_ALPHA"]))

    # index: src phrase -> list of (e_tokens, logphi, loglex)
    src_phrase_index: Dict[tuple, list] = {}
    for f, e_dict in phrase_table.items():
        cand = []
        for e, phi in e_dict.items():
            lp = math.log(max(phi, 1e-12))
            ll = math.log(max(lex_table.get(f, {}).get(e, 1e-12), 1e-12))
            cand.append((e, lp, ll))
        if cand:
            src_phrase_index[f] = cand

    NULL = "<NULL>"

    def decode_one(s_words: List[str]) -> List[str]:
        N = len(s_words)
        span_options = defaultdict(list)
        for i in range(N):
            for L in range(1, min(cfg["MAX_SRC_PHRASE_LEN"], N - i) + 1):
                f = tuple(s_words[i:i+L])
                if f in src_phrase_index:
                    span_options[(i, L)] = src_phrase_index[f]

        @lru_cache(maxsize=None)
        def search(mask: int, w1: str, w2: str):
            if mask == (1 << N) - 1:
                return ([], cfg["W_LM"] * lm_logprob(EOS, w1, w2))
            best_hyp, best_score = [], -1e9

            # leftmost uncovered position
            pos = 0
            while pos < N and ((mask >> pos) & 1):
                pos += 1

            advanced = False
            # place phrase at pos
            for L in range(1, min(cfg["MAX_SRC_PHRASE_LEN"], N - pos) + 1):
                if any(((mask >> k) & 1) for k in range(pos, pos+L)): 
                    continue
                key = (pos, L)
                if key not in span_options:
                    continue
                advanced = True
                new_mask = mask | sum(1 << k for k in range(pos, pos+L))
                for e_tokens, lp, ll in span_options[key]:
                    lm_s = 0.0
                    ww1, ww2 = w1, w2
                    for tok in e_tokens:
                        lm_s += lm_logprob(tok, ww1, ww2)
                        ww1, ww2 = ww2, tok
                    sub_hyp, sub_score = search(new_mask, ww1, ww2)
                    score = sub_score + cfg["W_PHRASE"]*lp + cfg["W_LEX"]*ll + cfg["W_LM"]*lm_s \
                            + cfg["W_WORD_PENALTY"]*len(e_tokens)
                    if score > best_score:
                        best_score = score
                        best_hyp = list(e_tokens) + sub_hyp

            # limited jump
            for jump in range(1, int(cfg["MAX_JUMP"]) + 1):
                jpos = pos + jump
                if jpos >= N: break
                if (mask >> jpos) & 1: continue
                for L in range(1, min(cfg["MAX_SRC_PHRASE_LEN"], N - jpos) + 1):
                    if any(((mask >> k) & 1) for k in range(jpos, jpos+L)):
                        continue
                    key = (jpos, L)
                    if key not in span_options:
                        continue
                    advanced = True
                    new_mask = mask | sum(1 << k for k in range(jpos, jpos+L))
                    for e_tokens, lp, ll in span_options[key]:
                        lm_s = 0.0
                        ww1, ww2 = w1, w2
                        for tok in e_tokens:
                            lm_s += lm_logprob(tok, ww1, ww2)
                            ww1, ww2 = ww2, tok
                        sub_hyp, sub_score = search(new_mask, ww1, ww2)
                        score = sub_score + cfg["W_PHRASE"]*lp + cfg["W_LEX"]*ll + cfg["W_LM"]*lm_s \
                                + cfg["W_WORD_PENALTY"]*len(e_tokens) + cfg["DIST_PENALTY"]*jump
                        if score > best_score:
                            best_score = score
                            best_hyp = list(e_tokens) + sub_hyp

            # backoff: IBM1 top-1 word
            if not advanced:
                s = s_words[pos]
                cands = ibm1.get(s, {})
                best = None
                for t, p in sorted(cands.items(), key=lambda kv: kv[1], reverse=True):
                    if t != NULL:
                        best = (t, p); break
                if best:
                    tok = best[0]
                    lm_s = lm_logprob(tok, w1, w2)
                    sub_hyp, sub_score = search(mask | (1 << pos), w2, tok)
                    lp = math.log(max(best[1], 1e-12))
                    ll = lp
                    score = sub_score + cfg["W_PHRASE"]*lp + cfg["W_LEX"]*ll + cfg["W_LM"]*lm_s + cfg["W_WORD_PENALTY"]
                    if score > best_score:
                        best_score = score
                        best_hyp = [tok] + sub_hyp

            return (best_hyp, best_score)

        hyp, _ = search(0, "<s>", "<s>")
        return hyp

    return decode_one


In [5]:
def compute_overlap_prf(ref_tok: List[List[str]], hyp_tok: List[List[str]]):
    overlap = ref_count = hyp_count = 0
    for r, h in zip(ref_tok, hyp_tok):
        cr = Counter(r); ch = Counter(h)
        ref_count += sum(cr.values())
        hyp_count += sum(ch.values())
        for w, c in ch.items():
            overlap += min(c, cr.get(w, 0))
    precision = overlap / hyp_count if hyp_count else 0.0
    recall    = overlap / ref_count if ref_count else 0.0
    f1 = 0.0 if (precision+recall)==0 else 2*precision*recall/(precision+recall)
    return precision, recall, f1

def evaluate(system_name: str, best_dir: str, test_path: str, out_csv: str=None) -> pd.DataFrame:
    # allow pointing to parent; auto-infer best/best_model
    if os.path.isdir(best_dir) and os.path.basename(best_dir).lower() in ["best", "best_model"]:
        pass
    elif os.path.isdir(os.path.join(best_dir, "best")):
        best_dir = os.path.join(best_dir, "best")
    elif os.path.isdir(os.path.join(best_dir, "best_model")):
        best_dir = os.path.join(best_dir, "best_model")

    src_tok, ref_tok, _, _ = load_test(test_path)
    decode = build_decoder(best_dir)

    hyp_tok, hyp_str = [], []
    for s in src_tok:
        h = decode(s)
        hyp_tok.append(h)
        hyp_str.append(" ".join(h))

    bleu = sacrebleu.corpus_bleu(hyp_str, [[" ".join(t) for t in ref_tok]])
    chrf = sacrebleu.corpus_chrf(hyp_str, [[" ".join(t) for t in ref_tok]])

    precision, recall, f1 = compute_overlap_prf(ref_tok, hyp_tok)
    exact_acc = sum(hs.strip()==" ".join(rt).strip() for hs, rt in zip(hyp_str, ref_tok)) / max(1, len(ref_tok))

    row = {
        "system": system_name,
        "bleu": round(bleu.score, 4),
        "chrf": round(chrf.score, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "exact_acc": round(exact_acc, 4),
    }
    df = pd.DataFrame([row], columns=["system","bleu","chrf","precision","recall","f1","exact_acc"])
    if out_csv:
        df.to_csv(out_csv, index=False, encoding="utf-8")
    return df


In [11]:
from pathlib import Path

# You're currently in ...\Artificial-Intelligent\model
ROOT = Path.cwd().parent                         # jump to project root
BEST = Path.cwd() / "smt_runs" / "zh_en_pbsmt_s2t_only" / "best"
TEST = ROOT / "data" / "dataset_CN_EN.txt"
OUT  = ROOT / "scores.csv"                       # optional

print("BEST exists? ", BEST.exists())
print("TEST exists? ", TEST.exists())

# Run the evaluation (evaluate() must already be defined from previous cell)
df = evaluate("SMT", str(BEST), str(TEST), str(OUT))
df


BEST exists?  True
TEST exists?  True


,system,bleu,chrf,precision,recall,f1,exact_acc
0,SMT,23.5836,50.4849,0.5681,0.5736,0.5709,0.0126
